# Raw To Bronze Ingest

Este notebook realiza a ingestão de dados da camada **Raw** para a camada **Bronze** de um medallion architecture no Databricks, utilizando o **Auto Loader** (`cloudFiles`) com streaming.

## Funcionalidades

- **Leitura incremental de arquivos CSV** a partir de um caminho configurável (`RAW_PATH`) via Auto Loader
- **Inferência automática de tipos** de colunas (`inferColumnTypes`)
- **Evolução de schema** com adição de novas colunas automaticamente (`addNewColumns`)
- **Coluna de dados resgatados** (`_rescued_data`) para capturar registros que não aderem ao schema
- **Colunas de auditoria**: origem do arquivo (`_source_file`) e timestamp de ingestão (`_ingestion_timestamp`)
- **Escrita em tabela Delta** com modo append e checkpoint para garantir o processamento exatamente uma vez

## Parâmetros (widgets)

| Parâmetro | Descrição |
|-----------|-----------|
| `ENVIRONMENT` | Ambiente de execução (dev, staging, prod) |
| `RAW_PATH` | Caminho dos arquivos brutos no storage |
| `CHECKPOINT_PATH` | Localização do checkpoint do stream |
| `SCHEMA_PATH` | Localização do schema do Auto Loader |
| `CATALOG` | Catálogo do Unity Catalog de destino |
| `SCHEMA` | Schema de destino |
| `TABLE_NAME` | Nome da tabela Bronze de destino |

## Fluxo

1. Define os widgets de configuração
2. Recupera os valores dos parâmetros
3. Cria o stream de leitura com Auto Loader e adiciona colunas de auditoria
4. Grava na tabela Bronze via `writeStream` com `trigger(availableNow=True)` (execução sob demanda)
5. Valida os dados com uma consulta de amostragem



In [0]:
from pyspark.sql import functions as F

dbutils.widgets.dropdown("ENVIRONMENT", "dev", ["dev", "staging", "prod"])
dbutils.widgets.text("RAW_PATH", "")
dbutils.widgets.text("CHECKPOINT_PATH", "")
dbutils.widgets.text("SCHEMA_PATH", "")
dbutils.widgets.text("CATALOG", "")
dbutils.widgets.text("SCHEMA", "")
dbutils.widgets.text("TABLE_NAME", "")


In [0]:
ENVIRONMENT = dbutils.widgets.get("ENVIRONMENT")
RAW_PATH = dbutils.widgets.get("RAW_PATH")
CHECKPOINT_PATH = dbutils.widgets.get("CHECKPOINT_PATH")
CATALOG = dbutils.widgets.get("CATALOG")
SCHEMA = dbutils.widgets.get("SCHEMA")
TABLE_NAME = dbutils.widgets.get("TABLE_NAME")
SCHEMA_PATH = dbutils.widgets.get("SCHEMA_PATH")

In [0]:
TARGET = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

spark.sql(f"DROP TABLE IF EXISTS {TARGET}")
dbutils.fs.rm(CHECKPOINT_PATH, True)
dbutils.fs.rm(SCHEMA_PATH, True)

In [0]:
import re

def clean_name(col: str) -> str:
    """Normaliza o nome da coluna: remove BOM/símbolos e converte para snake_case minúsculo.
    '$ Volume' é tratado como caso especial para manter o nome 'sales_volume'."""
    if col.replace("\ufeff", "").strip() == "$ Volume":
        return "sales_volume"
    return re.sub(r"[^0-9a-zA-Z]+", "_", col).strip("_").lower()

In [0]:
df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", SCHEMA_PATH)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("header", "true")
        .option("rescuedDataColumn", "_rescued_data")
    )

if TABLE_NAME == 'tb_sales':
    df = (
            df.option("encoding", "UTF-16LE")
           .option("sep", "\t")
           .option("lineSep", "\r\n")
        )
    

df = df.load(RAW_PATH)

In [0]:
if TABLE_NAME == 'tb_sales':
    df = (
        df.select(*[F.col(f"`{c}`").alias(c if c.startswith("_") else clean_name(c)) for c in df.columns])
    )



In [0]:

df = (
        df.withColumn("_source_file", F.col("_metadata.file_path"))
          .withColumn("_ingestion_timestamp", F.current_timestamp())
    )

query = (
        df.writeStream
        .format("delta")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .outputMode("append")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(f"{CATALOG}.{SCHEMA}.{TABLE_NAME}")
    )
query.awaitTermination()
